# QMI lookups and dashboard presets: SQL Server to Databricks

Read the six lookup tables and dashboard presets with `pd.read_sql(..., conn)` and copy them to
`integrated_operations_dev.analytics` using their existing names and columns.
SQL Server is read only. Existing BU IDs are copied as data, not regenerated.

Place `Upload.env` beside the notebook (the existing lowercase `upload.env` is also accepted).
Use the same `server`, `database`, `user`, `password`, and `schema` settings as the SQL uploader,
plus `DATABRICKS_HOST`, `DATABRICKS_TOKEN`, and `DATABRICKS_SQL_WAREHOUSE_ID`.
Uppercase SQL setting names are also accepted; `schema` defaults to `qmi`.
The SQL login must be able to read all seven source tables, including `ecosystem_source.qmi.costcenterkey` and the existing presets table. Set `PRESETS_SCHEMA` in Upload.env if that table is not under `dbo`. The preset table is migrated by key; later Databricks edits are preserved on rerun.

Install dependencies if needed:
```python
%pip install pandas pyodbc python-dotenv "databricks-sql-connector>=3.0.0"
```
ODBC Driver 18 for SQL Server must also be installed.

In [ ]:
from pathlib import Path
import pandas as pd
import pyodbc
from dotenv import dotenv_values
from IPython.display import display

env_path = Path("Upload.env")
if not env_path.is_file():
    env_path = Path("upload.env")
if not env_path.is_file():
    raise FileNotFoundError("Missing Upload.env beside the notebook.")
credentials = dotenv_values(env_path)

def setting(name):
    return credentials.get(name) or credentials.get(name.upper()) or ""

server = setting("server").strip()
database = setting("database").strip()
username = (setting("user") or setting("username")).strip()
password = setting("password")
schema = setting("schema").strip() or "qmi"
presets_schema = setting("PRESETS_SCHEMA").strip() or "dbo"

missing = [name for name, value in {
    "server": server, "database": database, "user": username, "password": password
}.items() if not value]
if missing:
    raise ValueError(f"Missing {env_path.name} settings: {missing}")

def odbc_value(value):
    return "{" + str(value).replace("}", "}}") + "}"

conn_str = (
    "DRIVER={ODBC Driver 18 for SQL Server};"
    f"SERVER={odbc_value(server)};DATABASE={odbc_value(database)};"
    f"ENCRYPT=yes;UID={odbc_value(username)};PWD={odbc_value(password)};"
    "TrustServerCertificate=YES;"
)
sql_schema = "[" + schema.replace("]", "]]") + "]"
presets_sql_schema = "[" + presets_schema.replace("]", "]]") + "]"

In [ ]:
# Keep the SQL reads in the same pandas / conn format as the other notebooks.
# These are SELECTs only; no SQL Server tables are changed.
conn = pyodbc.connect(conn_str, timeout=15)
try:
    costcenterkey = pd.read_sql("SELECT * FROM [ecosystem_source].[qmi].[costcenterkey]", conn, coerce_float=False)
    cost_element_key = pd.read_sql(f"SELECT * FROM {sql_schema}.[cost_element_key]", conn, coerce_float=False)
    cost_category_key = pd.read_sql(f"SELECT * FROM {sql_schema}.[cost_category_key]", conn, coerce_float=False)
    facilitygroupings = pd.read_sql("SELECT * FROM [qmi].[facilitygroupings]", conn, coerce_float=False)
    businessunits = pd.read_sql("SELECT * FROM [qmi].[businessunits]", conn, coerce_float=False)
    bugroupings = pd.read_sql("SELECT * FROM [qmi].[bugroupings]", conn, coerce_float=False)
    dashboard_presets = pd.read_sql(f"SELECT * FROM {presets_sql_schema}.[dashboard_presets]", conn, coerce_float=False)
finally:
    conn.close()

In [ ]:
tables = {
    "costcenterkey": costcenterkey,
    "cost_element_key": cost_element_key,
    "cost_category_key": cost_category_key,
    "facilitygroupings": facilitygroupings,
    "businessunits": businessunits,
    "bugroupings": bugroupings,
    "dashboard_presets": dashboard_presets,
}

# Preserve the integer BU relationship, including nullable integer handling.
businessunits["Id"] = businessunits["Id"].astype("Int64")
bugroupings["BusinessUnitId"] = bugroupings["BusinessUnitId"].astype("Int64")
if businessunits["Id"].isna().any() or businessunits["Id"].duplicated().any():
    raise ValueError("businessunits.Id must be populated and unique.")
if not bugroupings["BusinessUnitId"].isin(businessunits["Id"]).all():
    raise ValueError("bugroupings contains a BusinessUnitId missing from businessunits.")

# Keep SQL Server's UTC timestamps and numeric slot typed for the Delta MERGE.
required_preset_columns = {
    "MyID", "NetworkID", "UserName", "PresetSlot", "PresetName",
    "PresetState", "CreatedAt", "UpdatedAt"
}
missing_preset_columns = required_preset_columns - set(dashboard_presets.columns)
if missing_preset_columns:
    raise ValueError(f"dashboard_presets is missing columns: {sorted(missing_preset_columns)}")
dashboard_presets["PresetSlot"] = pd.to_numeric(
    dashboard_presets["PresetSlot"], errors="raise"
).astype("Int64")
for column in ("CreatedAt", "UpdatedAt"):
    dashboard_presets[column] = pd.to_datetime(
        dashboard_presets[column], utc=True
    ).dt.tz_localize(None)
if dashboard_presets[["MyID", "PresetSlot"]].isna().any().any():
    raise ValueError("dashboard_presets requires MyID and PresetSlot.")
if dashboard_presets.duplicated(["MyID", "PresetSlot"]).any():
    raise ValueError("dashboard_presets contains duplicate MyID/PresetSlot keys.")

display(pd.DataFrame([
    {"Table": name, "Rows": len(frame), "Columns": len(frame.columns),
     "Destination": f"integrated_operations_dev.analytics.{name}"}
    for name, frame in tables.items()
]))
for name, frame in tables.items():
    print(name)
    display(frame.head())

## Upload to Databricks

Set `RUN_DATABRICKS_UPLOAD = True` below to replace the six lookup tables and merge dashboard presets.
This follows the OTD and Safety upload format: parameterized batches, unique Delta staging
tables, row-count checks, and staging cleanup. The six lookups use
`CREATE OR REPLACE TABLE`; presets use a keyed `MERGE`.
Column mapping retains other source headers; `Cost Category ` is normalized to `Cost Category` during upload, even when only the upload cell is rerun.

All seven tables are prepared, staged, and checked before publication.
Empty lookup sources stop the upload; an empty preset source creates an empty destination.
Publication is atomic per table, **not across all seven**; if publication fails partway through,
rerun the upload cell. Lookups are replaced, while presets are merged by `MyID` and `PresetSlot`.
Only SQL preset rows newer than the Databricks row update it, so rerunning this notebook
does not erase presets saved in the app after migration. SQL identity, indexes, and foreign-key constraints
are not recreated in Databricks. Schema access is managed by the platform.
Each upload adds `last_updated`, a UTC timestamp shared by every row across all seven tables. It records when this upload was prepared, not when the SQL source row last changed.


In [ ]:
# Same opt-in switch as the existing OTD and Safety upload notebooks.
RUN_DATABRICKS_UPLOAD = False

if RUN_DATABRICKS_UPLOAD:
    import math
    import re
    import warnings
    from decimal import Decimal
    from uuid import uuid4
    from urllib.parse import urlparse
    from databricks import sql

    required = ["DATABRICKS_HOST", "DATABRICKS_TOKEN", "DATABRICKS_SQL_WAREHOUSE_ID"]
    settings = {name: (credentials.get(name) or "").strip() for name in required}
    missing = [name for name, value in settings.items() if not value]
    if missing:
        raise ValueError(f"Missing {env_path.name} settings: {missing}")

    host = settings["DATABRICKS_HOST"]
    parsed_host = urlparse(host if "://" in host else f"https://{host}")
    if (parsed_host.scheme != "https" or not parsed_host.hostname
            or parsed_host.username or parsed_host.password
            or parsed_host.path not in ("", "/") or parsed_host.query
            or parsed_host.fragment or parsed_host.port):
        raise ValueError("DATABRICKS_HOST must be a workspace hostname or HTTPS origin.")
    warehouse_id = settings["DATABRICKS_SQL_WAREHOUSE_ID"]
    if not re.fullmatch(r"[A-Za-z0-9_-]+", warehouse_id):
        raise ValueError("DATABRICKS_SQL_WAREHOUSE_ID must be the ID, not its HTTP path.")

    catalog, target_schema = "integrated_operations_dev", "analytics"

    def quote_identifier(value):
        return "`" + str(value).replace("`", "``") + "`"

    def column_type(series):
        dtype = series.dtype
        if pd.api.types.is_bool_dtype(dtype):
            return "BOOLEAN"
        if pd.api.types.is_integer_dtype(dtype):
            return "BIGINT"
        if pd.api.types.is_datetime64_any_dtype(dtype):
            return "TIMESTAMP"
        if pd.api.types.is_numeric_dtype(dtype):
            return "DOUBLE"
        values = series.dropna()
        if len(values) and all(isinstance(value, Decimal) for value in values):
            if not all(value.is_finite() for value in values):
                raise ValueError(f"Non-finite decimal in {series.name}.")
            scale = max(max(0, -value.as_tuple().exponent) for value in values)
            whole = max(max(0, value.adjusted() + 1) for value in values)
            precision = max(1, whole + scale)
            if precision > 38:
                raise ValueError(f"Decimal precision exceeds 38 in {series.name}.")
            return f"DECIMAL({precision},{scale})"
        return "STRING"

    def upload_value(data_type, value):
        if pd.isna(value):
            return None
        if data_type == "TIMESTAMP":
            return pd.Timestamp(value).to_pydatetime()
        if data_type == "BIGINT":
            return int(value)
        if data_type == "BOOLEAN":
            return bool(value)
        if data_type == "DOUBLE":
            number = float(value)
            if not math.isfinite(number):
                raise ValueError("Non-finite numeric lookup value.")
            return number
        if data_type.startswith("DECIMAL"):
            return value
        return str(value)

    # Prepare every table before making any Databricks changes.
    last_updated = pd.Timestamp.now(tz="UTC")
    prepared = []
    for name, frame in tables.items():
        if frame.empty and name != "dashboard_presets":
            raise ValueError(f"{name} has no rows; upload stopped before replacing any table.")
        frame = frame.copy()
        if name in ("cost_element_key", "cost_category_key"):
            # Normalize here so rerunning only this upload cell also fixes the table.
            frame.columns = [
                column.strip() if isinstance(column, str) and column.strip() == "Cost Category" else column
                for column in frame.columns
            ]
        frame["last_updated"] = last_updated
        columns = list(frame.columns)
        if len({str(column).casefold() for column in columns}) != len(columns):
            raise ValueError(f"Duplicate Databricks column names in {name}.")
        types = {column: column_type(frame[column]) for column in columns}
        prepared.append({
            "name": name,
            "target": ".".join(map(quote_identifier, [catalog, target_schema, name])),
            "staging": ".".join(map(quote_identifier, [catalog, target_schema, f"{name}_upload_{uuid4().hex}"])),
            "columns": ", ".join(map(quote_identifier, columns)),
            "definition": ", ".join(f"{quote_identifier(c)} {types[c]}" for c in columns),
            "placeholders": "(" + ", ".join("?" for _ in columns) + ")",
            "rows": [tuple(upload_value(types[c], value) for c, value in zip(columns, row))
                     for row in frame.itertuples(index=False, name=None)],
        })

    with sql.connect(
        server_hostname=parsed_host.hostname,
        http_path=f"/sql/1.0/warehouses/{warehouse_id}",
        access_token=settings["DATABRICKS_TOKEN"],
    ) as connection:
        with connection.cursor() as cursor:
            cursor.execute(f"DESCRIBE SCHEMA {quote_identifier(catalog)}.{quote_identifier(target_schema)}")
            cursor.fetchall()
            created_stages = []
            try:
                for item in prepared:
                    staging = item["staging"]
                    cursor.execute(
                        f"CREATE TABLE {staging} ({item['definition']}) USING DELTA "
                        "TBLPROPERTIES ('delta.columnMapping.mode' = 'name')"
                    )
                    created_stages.append(staging)
                    rows = item["rows"]
                    batch_size = 100
                    for offset in range(0, len(rows), batch_size):
                        batch = rows[offset:offset + batch_size]
                        placeholders = ", ".join(item["placeholders"] for _ in batch)
                        parameters = [value for row in batch for value in row]
                        cursor.execute(
                            f"INSERT INTO {staging} ({item['columns']}) VALUES {placeholders}",
                            parameters,
                        )
                    cursor.execute(f"SELECT COUNT(*) FROM {staging}")
                    staged_count = int(cursor.fetchone()[0])
                    if staged_count != len(rows):
                        raise RuntimeError(f"{item['name']}: staging count {staged_count} != {len(rows)}")
                    print(f"Staged {item['name']}: {staged_count:,} rows.")

                for item in prepared:
                    if item["name"] == "dashboard_presets":
                        # Merge the SQL snapshot without overwriting more recent app saves.
                        cursor.execute(
                            f"CREATE TABLE IF NOT EXISTS {item['target']} ({item['definition']}) "
                            "USING DELTA TBLPROPERTIES ('delta.columnMapping.mode' = 'name')"
                        )
                        cursor.execute(
                            f"MERGE INTO {item['target']} AS target "
                            f"USING {item['staging']} AS source "
                            "ON target.`MyID` = source.`MyID` "
                            "AND target.`PresetSlot` = source.`PresetSlot` "
                            "WHEN MATCHED AND (target.`UpdatedAt` IS NULL "
                            "OR source.`UpdatedAt` > target.`UpdatedAt`) "
                            "THEN UPDATE SET * "
                            "WHEN NOT MATCHED THEN INSERT *"
                        )
                        cursor.execute(f"SELECT COUNT(*) FROM {item['target']}")
                        after = int(cursor.fetchone()[0])
                        if after < len(item["rows"]):
                            raise RuntimeError(
                                f"dashboard_presets: published count {after} is less than SQL source "
                                f"count {len(item['rows'])}"
                            )
                        print(f"Merged {catalog}.{target_schema}.dashboard_presets: "
                              f"{after:,} total rows.")
                    else:
                        cursor.execute(
                            f"CREATE OR REPLACE TABLE {item['target']} USING DELTA "
                            "TBLPROPERTIES ('delta.columnMapping.mode' = 'name') "
                            f"AS SELECT {item['columns']} FROM {item['staging']}"
                        )
                        cursor.execute(f"SELECT COUNT(*) FROM {item['target']}")
                        after = int(cursor.fetchone()[0])
                        if after != len(item["rows"]):
                            raise RuntimeError(
                                f"{item['name']}: published count {after} != {len(item['rows'])}"
                            )
                        print(f"Updated {catalog}.{target_schema}.{item['name']}: "
                              f"{after:,} rows.")
            finally:
                for staging in created_stages:
                    try:
                        cursor.execute(f"DROP TABLE IF EXISTS {staging}")
                    except Exception:
                        warnings.warn(f"Could not remove staging table {staging}; remove it when the connection is available.")
else:
    print("Databricks upload disabled. Set RUN_DATABRICKS_UPLOAD = True to replace the six dev lookups and merge dashboard presets.")